In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 02a_rq1_qual_analysis.py
# Purpose of Script: Identify all synthetic media in Qualitative SOR data.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Install Packages
#~~~~~~~~~~~~~~~~~~~~~~~~~~
!pip install langdetect
!pip install deep_translator
!pip install pycountry

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=dbc6fe245ed1df01d3e2cee1e84ebfe671b5c504dfe2d2a2f6148337ebc3620d
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 36.8 MB/s eta 0:00:00


In [4]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from deep_translator import GoogleTranslator
import pycountry
import re
from pathlib import Path

In [5]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [6]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect("/content/working.duckdb")

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_clean = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Functions ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Language Detection
def detect_language(text):
  if not isinstance(text, str) or text.strip() == "":
    return "No Language"
  try:
    return detect(text)
  except LangDetectException:
    return "No Language"

# Long Form Language Name
def obtain_lf_language(code):
  code = code.split("-")[0]
  language = pycountry.languages.get(alpha_2=code)
  if language:
    return language.name
  return "Unknown"

In [7]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Qualitative Analysis ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Motivation: Search Qualitative Analysis for Synthethic Media
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Qualitative Analysis ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_qual = con.execute(f""" select * from '{path_samp}qual_final.parquet'""").df()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Data Quality Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_dq = pd.read_csv(f"{path_out}01_dq_1_overview.csv")
df_dq = df_dq[["platform","files","total_rows"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
df_qual[(df_qual["platform"] == "tiktok") & (df_qual["q"] == "incompatible_content_explanation")].head(20)

,platform,date,q_id,q,statement,total,version
888,tiktok,2025-01-01,821,incompatible_content_explanation,while adults make personal choices about how t...,258453.0,20260724_final
889,tiktok,2025-01-01,822,incompatible_content_explanation,we welcome the respectful expression of differ...,934413.0,20260724_final
890,tiktok,2025-01-01,823,incompatible_content_explanation,tiktok is enriched by the various backgrounds ...,158137.0,20260724_final
891,tiktok,2025-01-01,824,incompatible_content_explanation,we maintain content eligibility standards for ...,754085.0,20260724_final
892,tiktok,2025-01-01,825,incompatible_content_explanation,your content is against our community guidelin...,50150.0,20260724_final
893,tiktok,2025-01-01,826,incompatible_content_explanation,your content is against our community guidelin...,59155.0,20260724_final
894,tiktok,2025-01-01,827,incompatible_content_explanation,we welcome the respectful expression of differ...,174.0,20260724_final
895,tiktok,2025-01-01,828,incompatible_content_explanation,we celebrate all shapes and sizes and want peo...,76336.0,20260724_final
896,tiktok,2025-01-01,829,incompatible_content_explanation,,65.0,20260724_final
897,tiktok,2025-01-01,830,incompatible_content_explanation,ton live n'est pas éligible pour les recommand...,1758.0,20260724_final


In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Find Unique Q/Statement Combinations ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Frequencies of Statements - All Time
out_qual_freq_all = (df_qual.groupby(
    ["platform","q","statement"], as_index=False)["total"].sum())

out_qual_freq_all["platform"] = out_qual_freq_all["platform"].str.capitalize()

#### Join Total Rows
out_qual_freq_all = out_qual_freq_all.merge(df_dq[["platform","total_rows"]], on="platform",how="left")

#### Relative Frequency
out_qual_freq_all["rel_freq"] = (out_qual_freq_all["total"]/out_qual_freq_all["total_rows"]*100).round(2)

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Find Unique Number of Statements ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
unique_statements = df_qual.groupby(["platform","q"],
                                    as_index=False).agg(unique_responses=("statement","nunique"))

unique_statements["platform"] = unique_statements["platform"].str.capitalize()

#### Add Total Unique Per Platform Rows
total_per_platform = unique_statements.groupby("platform", as_index=False).agg(unique_responses=("unique_responses","sum"))
total_per_platform["q"] = ""
unique_statements_out = pd.concat([unique_statements, total_per_platform], ignore_index=True)

#### Add Total Row
total_row = pd.DataFrame({"platform" : "All Platforms",
                          "q" : "",
                          "unique_responses" : [unique_statements["unique_responses"].sum()]})

unique_statements_out = pd.concat([unique_statements_out, total_row], ignore_index=True)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
out_qual_freq_all.to_csv(f"{path_out}02_rq1_qual_1_statement_frequencies.csv")
unique_statements_out.to_csv(f"{path_out}02_rq1_qual_2_number_unique_responses.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Identifying Potential Synthetic Media ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Motivation - Can qualitative statements reveal moderation of any synthetic
# media? Pipeline - translate all statements into english, then systematically
# search for references to AI generated/ altered material.
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Translate to English ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Cut Down Sample
df_qual_cut = df_qual[["platform","q","statement"]].copy().drop_duplicates()

In [ ]:
### Define Statement Language - Approx 18 minute calculation
df_qual_cut["statement_lang"] = df_qual_cut["statement"].apply(detect_language)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Temp Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Save Temporary Language Detection Results
df_qual_cut.to_csv(f"{path_out}02_rq1_qual_3_temp_lang_detection_results.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Temp Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Temporary Language Detection Results
## Allowing broken run of script
df_qual_cut = pd.read_csv(f"{path_out}02_rq1_qual_3_temp_lang_detection_results.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Lanuage Distribution ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Assess distribution of languages - ALL
lang_freq = pd.DataFrame(df_qual_cut["statement_lang"].value_counts())

# Assess distribution of languages - Platform Level
lang_freq_plat = (df_qual_cut.groupby(['platform','statement_lang']).size().reset_index(name='count'))

### Platform Level - Horizontal Lanuage Table
lang_freq_plat_cross = pd.crosstab(lang_freq_plat["platform"], lang_freq_plat["statement_lang"])

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Assess Accuracy of Lanuage Detection Algorithm ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
## Merge Total Unique Statements
lang_freq_plat["platform"] = lang_freq_plat["platform"].str.capitalize()
unique_statements_to_join = unique_statements_out[unique_statements_out["q"] == ""]

lang_freq_plat = lang_freq_plat.merge(unique_statements_to_join[["platform","unique_responses"]],
                                      on="platform", how="left")

## Relative Language Frequency Column
lang_freq_plat["rel_freq"] = (lang_freq_plat["count"]/lang_freq_plat["unique_responses"]*100).round(2)

## Obtain Language Long Form Name
lang_freq_plat["language_name"] = lang_freq_plat["statement_lang"].apply(obtain_lf_language)

## Reorder
lang_freq_plat = lang_freq_plat[["platform","statement_lang","language_name","count","unique_responses","rel_freq"]]
lang_freq_plat = lang_freq_plat.rename(columns={"unique_responses" : "total_unique_responses"})

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Lanuage Analysis Results ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
lang_freq_plat.to_csv(f"{path_out}02_rq1_qual_4_language_detection_analysis_part_1.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Manually Check Lanuage Detection ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
## Isolate Rows Where No Language Was Detected
df_no_language_detected = df_qual_cut[df_qual_cut["statement_lang"] == "No Language"].reset_index()

## Export Intermediate Step
df_no_language_detected.to_csv(f"{path_out}02_rq1_qual_5_qualitative_statements_with_no_language_detected.csv")

## How Many Statements Couldn't Be Dectected? - 176/214442 - 0.08%
lang_freq_plat["count"].sum()
lang_freq_plat[lang_freq_plat["statement_lang"] == "No Language"]["count"].sum()

np.int64(176)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Fix Incorrect Detections ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# All 176 Detections Classified as 'No Langage' are all URLs or '0' or ' '
# Manually Overwrite these to English in Analysis Tables
# Follow Steps in Previous Code to Obtain Frequency Table with Manual Corrections
lang_freq_plat_man_fix = (df_qual_cut.groupby(['platform','statement_lang']).size().reset_index(name='count'))
lang_freq_plat_man_fix["statement_lang"] = lang_freq_plat_man_fix["statement_lang"].replace("No Language","en")
lang_freq_plat_man_fix = (lang_freq_plat_man_fix.groupby(['platform','statement_lang'], as_index=False).agg(count=("count","sum")))
lang_freq_plat_man_fix["platform"] = lang_freq_plat_man_fix["platform"].str.capitalize()
lang_freq_plat_man_fix = lang_freq_plat_man_fix.merge(unique_statements_to_join[["platform","unique_responses"]],
                                      on="platform", how="left")
lang_freq_plat_man_fix["rel_freq"] = (lang_freq_plat_man_fix["count"]/lang_freq_plat_man_fix["unique_responses"]*100).round(2)
lang_freq_plat_man_fix["language_name"] = lang_freq_plat_man_fix["statement_lang"].apply(obtain_lf_language)
lang_freq_plat_man_fix = lang_freq_plat_man_fix[["platform","statement_lang","language_name","count","unique_responses","rel_freq"]]
lang_freq_plat_man_fix = lang_freq_plat_man_fix.rename(columns={"unique_responses" : "total_unique_responses"})

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Lanuage Analysis Results - After Manual Patching ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
lang_freq_plat_man_fix.to_csv(f"{path_out}02_rq1_qual_6_language_detection_analysis_after_manual_patch.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Translate Statements ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Isolate Subset to Translate
df_to_translate = df_qual_cut.copy()

# Manually Patch Incorrect Language Detection
df_to_translate["statement_lang"] = df_to_translate["statement_lang"].replace("No Language","en")

# Find Entries Where Language is not English - 4332 Entries Identified
df_to_translate = df_to_translate[df_to_translate['statement_lang'] != 'en'].reset_index(drop=True)
df_to_translate = df_to_translate.sort_values(['platform','statement_lang'], ascending=[True, False]).reset_index(drop=True)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Assess Incorrect Language Assignments ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# 4332 Statements Assigned as being Non-English
## Manually Check - Export to Excel
df_to_translate.to_csv(f"{path_out}02_rq1_qual_7_manually_check_incorrect_language_assignments.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Flagged Incorrect Language Assignments ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_to_translate_checked = pd.read_excel(f"{path_out}02_rq1_qual_8_manually_checked_language_assignments.xlsx")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Analysis of Incorrectly Assigned Lanugages ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
to_translate_summary = (df_to_translate_checked.groupby("platform",
                                                        as_index=False).agg(total_rows=("incorrect_flag","size"),
                                                                            incorrect=("incorrect_flag","sum")))

to_translate_summary["incorrect_perc"] = (to_translate_summary["incorrect"]/to_translate_summary["total_rows"]*100).round(2)
to_translate_summary["platform"] = to_translate_summary["platform"].str.capitalize()
to_translate_summary = to_translate_summary.sort_values("incorrect_perc")

### Total Non-English Statements - 4332
### Only TikTok Statements put forward for Translation

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~
to_translate_summary.to_csv(f"{path_out}02_rq1_qual_9_translation_error_summary_table.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Perform Translation
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Isolate Data to be Translated
df_to_translate_table = df_to_translate_checked.copy().reset_index(drop=True).drop(columns=["Column1"])
df_to_translate_table = df_to_translate_table[df_to_translate_table["incorrect_flag"] == 0]
df_to_translate_table = df_to_translate_table.drop(columns=["incorrect_flag_manual","incorrect_flag"]).reset_index(drop=True)
df_to_translate_table["platform"] = df_to_translate_table["platform"].str.capitalize()

In [ ]:
# Set-Up Translator
translator = GoogleTranslator(source = "auto", target = "en")

In [ ]:
# Translate - 3498 Statements to Translate - 45 minutes to execute
df_to_translate_table['statement_en'] = df_to_translate_table['statement'].apply(lambda x: translator.translate(x))

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Translated Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_to_translate_table.to_csv(f"""{path_out}02_rq1_qual_10_translated_statements.csv""")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Translated Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_trans = pd.read_csv(f"{path_out}02_rq1_qual_10_translated_statements.csv")

In [ ]:
# Join Translations to Unique Statements
df_qual_cut["platform"] = df_qual_cut["platform"].str.capitalize()
df_complete = df_qual_cut.merge(df_trans[["platform","q","statement","statement_en"]].drop_duplicates(),
                             on=["platform","q","statement"],
                             how="left")

In [ ]:
# Assign New Statement Column Based on Populated Translation
df_complete["statement_final"] = df_complete["statement_en"].fillna(df_complete["statement"])

# Format
df_complete = df_complete.drop(columns=["statement_lang","statement_en"])

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Synthetic Media Search ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Create List of Candidate Words
#~~~~~~~~~~~~~~~~~~~~~~~~~~
synth_keywords = [# synthetic
                  "synthetic", "syn", "synt",
                  # ai
                  "artificial", "ai-g", "generative", "genai", "ai gener", "aigc",
                  # deepfake
                  "deepfake", "deep fake", "deep",
                  # manipulated media/ digitally enhances
                  "manipulated", "manip",
                  # edited
                  "edited", "altered", "modified",  "digitally",
                  # fake
                  "fake", "fabri", "fabrication",
                  # cloned
                  "cloned", "clone",
                  # impersonation
                  "imperson",
                  # large language
                  "large lan"]

# Build Expression
pattern = re.compile("|".join(re.escape(p) for p in synth_keywords), re.IGNORECASE)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Search ~4m
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_complete["possible_synth_flag"] = df_complete["statement_final"].str.contains(pattern,na=False)

# Match Keyword
df_complete["keyword_match"] = df_complete["statement_final"].str.extract(f"({pattern.pattern})",
                                                                          flags=re.IGNORECASE,
                                                                          expand=False)

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Keyword Match Frequency
#~~~~~~~~~~~~~~~~~~~~~~~~~~
keyword_freq = (df_complete["keyword_match"].value_counts(dropna=False).reset_index())

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Potentially Synthetic Statements ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Manually Categorize into Confirmed and Rejected Synthetic Media in Qualitative Statements
# Decisions Made in xlsx File
df_potential = df_complete[df_complete["possible_synth_flag"] == True].reset_index(drop=True)
df_potential = df_potential.sort_values(["platform","keyword_match","statement_final"],ascending=True).reset_index(drop=True)
df_potential.to_csv(f"{path_out}02_rq1_qual_11_potentially_synthetic.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Potentially Synthetic Statements - Post Decision ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Notes: 'Manipulated Media' in TikTok submissions is not strong enough to
# achieve being flagged as synthetic.
# Only TikTok has cleared labelled Synthetic Media Moderation
df_syn = pd.read_excel(f"{path_out}02_rq1_qual_12_synthetic_media_flagged.xlsx")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Identify Synthetic Media in Base Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Generate All Combinations Needed to Search Through Cleaned SOR data and to
# Extract Synthetic Media
### Cut Down Import
df_syn = df_syn[["platform","q","statement","keyword_match",
                 "ai_related_flag","synth_flag"]]

df_syn["platform"] = df_syn["platform"].str.lower()

In [ ]:
### Join to Original Qualitative Statement Data
df_rq1_syn = df_qual.merge(df_syn,
                           on=["platform","q","statement"],
                           how="left")

In [ ]:
### Format Columns After Join
df_rq1_syn = df_rq1_syn.fillna({"keyword_match" : "",
                                "ai_related_flag" : 0,
                                "synth_flag": 0})

In [ ]:
# Format
df_rq1_syn = df_rq1_syn[["platform","date","q_id","q","statement",
                         "total", "keyword_match", "ai_related_flag",
                         "synth_flag"]].reset_index(drop=True)

In [ ]:
# Export
df_rq1_syn.to_csv(f"{path_out}02_rq1_qual_13_synthetic_media_data.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Identify Patterns in Qualitative Statements ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Isolate Solid Synthetic Media Moderation
df_rq1 = df_rq1_syn[df_rq1_syn["synth_flag"] == 1].reset_index(drop=True)

# Sort
df_rq1 = df_rq1.sort_values(by=["platform","date","q","statement"])

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# TikTok
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# General Pattern in synthetic media is 'incompatible_content_explanation' is textually
# naunced. 'Incompataible_content_ground' contains the umbrella statement.
# Sum of Incompatiable Content Ground = Sum of Incompatible Content Explanation (Mostly)
# Check This:
check = (df_rq1[df_rq1["platform"] == "tiktok"]
         .query("q in ['incompatible_content_ground','incompatible_content_explanation']")
         .groupby(["date","q"])["total"]
         .sum()
         .unstack(fill_value=0))

check["equal"] = (check["incompatible_content_ground"] == check["incompatible_content_explanation"])
check["more_ground_than_explanation"] = (check["incompatible_content_ground"] > check["incompatible_content_explanation"])
check["more_explanation_than_ground"] = (check["incompatible_content_explanation"] > check["incompatible_content_ground"])

# Not always True - Build Query System to Search Cleaned Data - Below

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SQL Queries Base Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#### TikTok
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_qual_tiktok = df_rq1[(df_rq1["platform"] == "tiktok") & (df_rq1["synth_flag"] == 1)]
df_qual_tiktok = df_qual_tiktok[["date", "q_id", "q"]]

#### Define Dynamic SQL Lookup
lookup_daily = (df_qual_tiktok
                .groupby(["date", "q"])["q_id"]
                .apply(list)
                .unstack(fill_value=[])
                .reset_index())

lookup_daily["date"] = pd.to_datetime(lookup_daily["date"]).dt.strftime("%Y-%m-%d")

#### Batch Process
batch_size = 90
lookup_daily["batch"] = (lookup_daily.index // batch_size)

#### Define Path
parquet_root = Path(f"{path_clean}")

#### Drop Table If Exists
con.execute(f""" drop table if exists results""")

#### Loop Through Each Batch
#for batch in lookup_daily["batch"].unique():
for batch in [4]:

  subset = lookup_daily[lookup_daily["batch"] == batch]

  #### Initialise
  first = True

  #### Loop Through Each Cleaned File
  for _, row in subset.iterrows():

    # Extract Date, Ground Explanation, Content Ground
    date = row["date"]
    explanation_ids = row["incompatible_content_explanation"]
    ground_ids = row["incompatible_content_ground"]

    # Dynamically Define File Path
    parquet_file = parquet_root / f"{date}-tiktok" / "tiktok.parquet"

    # Dynically Generate Query Conditions
    explanation_sql = ",".join(map(str, explanation_ids))
    ground_sql = ",".join(map(str, ground_ids))

    # Dynamically Generate Query
    if first:
      query = f""" create table results as
                 select date, aut_det, aut_dec, cont_type, source,
                 cat, cat_spec, cat_spec_other, des_ground, des_fact,
                 illegal_c_ground, illegal_c_ex, incomp_c_ground,
                 incomp_c_ex, incomp_c_illegal,
                 des_vis, des_vis_other, des_vis_end_date,
                 des_mon, des_mon_other, des_mon_end_date,
                 des_prov, des_prov_end_date,
                 des_acc, des_acc_end_date
                 from read_parquet('{parquet_file}')
                 where incomp_c_ground in ({ground_sql})
                 or incomp_c_ex in ({explanation_sql})"""

      # Re-set Initial Temp Table Set-Up
      first = False

    else:
      query = f""" insert into results
                 select date, aut_det, aut_dec, cont_type, source,
                 cat, cat_spec, cat_spec_other, des_ground, des_fact,
                 illegal_c_ground, illegal_c_ex, incomp_c_ground,
                 incomp_c_ex, incomp_c_illegal,
                 des_vis, des_vis_other, des_vis_end_date,
                 des_mon, des_mon_other, des_mon_end_date,
                 des_prov, des_prov_end_date,
                 des_acc, des_acc_end_date
                 from read_parquet('{parquet_file}')
                 where incomp_c_ground in ({ground_sql})
                 or incomp_c_ex in ({explanation_sql})"""

    # Print Progress
    print(f"Processing TikTok Date: {date}")

    # Execute Query
    rel = con.execute(query)

    print(f"Finished TikTok Date: {date}")
    del rel

  # Save Results
  con.execute(f""" copy results
                     to '{path_out}02_rq1_qual_14_tiktok_synth_data_batch_{batch}.parquet'
                     (format parquet)""")

  # Clean Results
  con.execute(f""" drop table results""")
  con.commit()
  con.close()
  con = duckdb.connect("/content/working.duckdb")


In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Check Output ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_test = con.execute(f"select * from '{path_out}02_rq1_qual_14_tiktok_synth_data_batch_4.parquet' limit 10").df()